# Cosmos DB Parallel Container Migration Validation (Fabric)

This notebook orchestrates **parallel validation** of multiple Cosmos DB container migrations by reading a CSV configuration file and calling `CosmosDBLiveContainerMigrationValidation` for each container pair.

**How it works:**
| Step | Description |
|------|-------------|
| 1 | Read CSV with container migration definitions |
| 2 | Configure parallelism |
| 3 | Build DAG and execute parallel validation notebook runs via `notebookutils.notebook.runMultiple()` |
| 4 | Collect and display results |

In [6]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Cosmos_Migration"
    }
}

StatementMeta(, c8cff035-7370-4f42-bdbe-a40c3bb85899, -1, Finished, Available, Finished, True)

## Step 1: Read Migration Configuration CSV

Reads the same `cosmosDBLiveMigrationList.csv` used by the parallel migration notebook.

In [ ]:
# Read the migration configuration CSV from Lakehouse Files
csv_path = "Files/cosmosDBLiveMigrationList.csv"

migration_df = spark.read.option("header", True).csv(csv_path)

# Collect as list of dicts for building DAG activities
migration_list = [row.asDict() for row in migration_df.collect()]

# Display a formatted summary (masking keys)
def mask_key(val):
    if not val or len(val) < 8:
        return "****"
    return val[:4] + "..." + val[-4:]

print(f"Found {len(migration_list)} container migration(s) to validate:")
print("=" * 90)
for i, row in enumerate(migration_list):
    print(f"\n  [{i}] {row.get('cosmosSourceDatabaseName','?')}/{row.get('cosmosSourceContainerName','?')}")
    print(f"      -> {row.get('cosmosTargetDatabaseName','?')}/{row.get('cosmosTargetContainerName','?')}")
    print(f"      Source endpoint:    {row.get('cosmosSourceEndpoint','?')}")
    print(f"      Source key:         {mask_key(row.get('cosmosSourceMasterKey',''))}")
    print(f"      Target endpoint:    {row.get('cosmosTargetEndpoint','?')}")
    print(f"      Target key:         {mask_key(row.get('cosmosTargetMasterKey',''))}")
print("\n" + "=" * 90)

## Step 2: Configure Parallelism

Set the maximum number of validations to run in parallel.

> **Recommendation:** Start with 4 and adjust based on cluster capacity.

In [ ]:
# Maximum number of validations to run in parallel
num_notebooks_in_parallel = 4

# Name of the child notebook to call for each container validation
notebook_to_run = "CosmosDBLiveContainerMigrationValidation"

print(f"Will run up to {num_notebooks_in_parallel} validations in parallel")
print(f"Child notebook: {notebook_to_run}")
print(f"Total containers to validate: {len(migration_list)}")

## Step 3: Build DAG & Execute Parallel Validations

Builds a DAG of parallel activities from the CSV rows and executes them using `notebookutils.notebook.runMultiple()`. Each activity calls `CosmosDBLiveContainerMigrationValidation` with the source and target parameters for that container pair.

In [ ]:
# Build DAG activities from CSV migration list
activities = []
for i, row in enumerate(migration_list):
    activity = {
        "name": f"Validate_{i}_{row['cosmosSourceDatabaseName']}_{row['cosmosSourceContainerName']}",
        "path": notebook_to_run,
        "timeoutPerCellInSeconds": 600,
        "retry": 1,
        "retryIntervalInSeconds": 30,
        "args": {
            "cosmosSourceEndpoint": row["cosmosSourceEndpoint"],
            "cosmosSourceMasterKey": row["cosmosSourceMasterKey"],
            "cosmosTargetEndpoint": row["cosmosTargetEndpoint"],
            "cosmosTargetMasterKey": row["cosmosTargetMasterKey"],
            "cosmosRegion": row["cosmosRegion"],
            "cosmosSourceDatabaseName": row["cosmosSourceDatabaseName"],
            "cosmosSourceContainerName": row["cosmosSourceContainerName"],
            "cosmosTargetDatabaseName": row["cosmosTargetDatabaseName"],
            "cosmosTargetContainerName": row["cosmosTargetContainerName"],
        }
    }
    activities.append(activity)

# Build the DAG
dag = {
    "activities": activities,
    "timeoutInSeconds": 86400,
    "concurrency": num_notebooks_in_parallel
}

# Validate DAG
print(f"DAG built with {len(activities)} activities:")
for act in activities:
    print(f"  - {act['name']}")

is_valid = notebookutils.notebook.validateDAG(dag)
print(f"\nDAG validation: {'PASSED' if is_valid else 'FAILED'}")

# Execute all container validations in parallel
import json

print(f"Starting parallel validation of {len(activities)} containers (max {num_notebooks_in_parallel} concurrent)...")
print("=" * 90)

results = notebookutils.notebook.runMultiple(dag)

# ---------------------------------------------------------------------------
# Parse and display detailed results from child notebooks
# ---------------------------------------------------------------------------
print("\n" + "=" * 90)
print("VALIDATION RESULTS")
print("=" * 90)

pass_count = 0
fail_count = 0
error_count = 0
summary_rows = []

for activity_name, result in results.items():
    if result.get("exception"):
        error_count += 1
        print(f"\n  \u274c ERROR - {activity_name}")
        print(f"          Exception: {result['exception']}")
        summary_rows.append({
            "activity": activity_name, "status": "ERROR",
            "source": "?", "target": "?",
            "source_count": "?", "target_count": "?", "missing": "?",
        })
    else:
        exit_val = result.get("exitVal", "")
        try:
            detail = json.loads(exit_val)
            status = detail.get("status", "?")
            src = f"{detail.get('source_db','?')}/{detail.get('source_container','?')}"
            tgt = f"{detail.get('target_db','?')}/{detail.get('target_container','?')}"
            s_count = detail.get("source_count", "?")
            t_count = detail.get("target_count", "?")
            missing = detail.get("missing_count", "?")

            icon = "\u2705" if status == "PASS" else "\u274c"
            if isinstance(s_count, int):
                print(f"\n  {icon} {status} - {src} -> {tgt}")
                print(f"          Source: {s_count:,}  Target: {t_count:,}  Missing/changed: {missing:,}")
            else:
                print(f"\n  {icon} {status} - {src} -> {tgt}")
                print(f"          Source: {s_count}  Target: {t_count}  Missing/changed: {missing}")

            if status == "PASS":
                pass_count += 1
            else:
                fail_count += 1

            summary_rows.append({
                "activity": activity_name, "status": status,
                "source": src, "target": tgt,
                "source_count": s_count, "target_count": t_count, "missing": missing,
            })
        except (json.JSONDecodeError, TypeError):
            # Child notebook did not return structured JSON - show raw exit value
            pass_count += 1
            print(f"\n  \u2705 {activity_name}: {exit_val}")
            summary_rows.append({
                "activity": activity_name, "status": "OK (raw)",
                "source": "?", "target": "?",
                "source_count": "?", "target_count": "?", "missing": "?",
            })

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 90)
print(f"SUMMARY: {pass_count} passed, {fail_count} failed, {error_count} errors "
      f"out of {len(results)} total")

if fail_count > 0 or error_count > 0:
    print("\n  Containers needing attention:")
    for row in summary_rows:
        if row["status"] not in ("PASS", "OK (raw)"):
            print(f"    - {row['source']} -> {row['target']}  ({row['status']}, missing={row['missing']})")

print("=" * 90)

In [ ]:
# Execute all container validations in parallel
import json

print(f"Starting parallel validation of {len(activities)} containers (max {num_notebooks_in_parallel} concurrent)...")
print("=" * 90)

results = notebookutils.notebook.runMultiple(dag)

# ---------------------------------------------------------------------------
# Parse and display detailed results from child notebooks
# ---------------------------------------------------------------------------
print("\n" + "=" * 90)
print("VALIDATION RESULTS")
print("=" * 90)

pass_count = 0
fail_count = 0
error_count = 0
summary_rows = []

for activity_name, result in results.items():
    if result.get("exception"):

        error_count += 1print("=" * 90)

        print(f"\n  \u274c ERROR - {activity_name}")

        print(f"          Exception: {result['exception']}")            print(f"    - {row['source']} -> {row['target']}  ({row['status']}, missing={row['missing']})")

        summary_rows.append({        if row["status"] not in ("PASS", "OK (raw)"):

            "activity": activity_name, "status": "ERROR",    for row in summary_rows:

            "source": "?", "target": "?",    print("\n  Containers needing attention:")

            "source_count": "?", "target_count": "?", "missing": "?",if fail_count > 0 or error_count > 0:

        })

    else:      f"out of {len(results)} total")

        exit_val = result.get("exitVal", "")print(f"SUMMARY: {pass_count} passed, {fail_count} failed, {error_count} errors "

        try:print("\n" + "=" * 90)

            detail = json.loads(exit_val)# ---------------------------------------------------------------------------

            status = detail.get("status", "?")# Summary

            src = f"{detail.get('source_db','?')}/{detail.get('source_container','?')}"# ---------------------------------------------------------------------------

            tgt = f"{detail.get('target_db','?')}/{detail.get('target_container','?')}"

            s_count = detail.get("source_count", "?")            })

            t_count = detail.get("target_count", "?")                "source_count": "?", "target_count": "?", "missing": "?",

            missing = detail.get("missing_count", "?")                "source": "?", "target": "?",

                "activity": activity_name, "status": "OK (raw)",

            icon = "\u2705" if status == "PASS" else "\u274c"            summary_rows.append({

            if isinstance(s_count, int):            print(f"\n  \u2705 {activity_name}: {exit_val}")

                print(f"\n  {icon} {status} - {src} -> {tgt}")            pass_count += 1

                print(f"          Source: {s_count:,}  Target: {t_count:,}  Missing/changed: {missing:,}")            # Child notebook didn't return structured JSON — show raw exit value

            else:        except (json.JSONDecodeError, TypeError):

                print(f"\n  {icon} {status} - {src} -> {tgt}")            })

                print(f"          Source: {s_count}  Target: {t_count}  Missing/changed: {missing}")                "source_count": s_count, "target_count": t_count, "missing": missing,

                "source": src, "target": tgt,

            if status == "PASS":                "activity": activity_name, "status": status,

                pass_count += 1            summary_rows.append({

            else:
                fail_count += 1